# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset-level metadata as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and included fields/columns.

We'll inspect the schema to enumerate all top-level record sets, along with their `@id` and field structure (also shown with their `@id`).

In [ ]:
# Get all available record sets with their @id and fields/columns
print('Available Record Sets:')
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields'):
        field_ids = [f.id for f in rs.fields]
        print(f"  Field @ids: {field_ids}")
    elif hasattr(rs, 'columns'):
        # Some datasets may call them columns by the spec.
        column_ids = [c.id for c in rs.columns]
        print(f"  Column @ids: {column_ids}")
    else:
        print("  No fields or columns detected.")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis using their `@id`. We'll demonstrate using the first available record set.

In [ ]:
# Prepare list of all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Extract records into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    # Each record set yields a sequence of record dicts
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} rows for record set {rs_id}")

# For demonstration, pick the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nFirst record set: {first_rs}")
    print("Columns available:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate data processing: filtering, normalization, grouping.

We'll select a numeric column from the first record set for filtering (>10), normalization (z-score), and, if present, group by a categorical column. Please update the variables below based on the columns printed above.

In [ ]:
# Set your target record set and field ids below
# As an example, adjust these after inspecting the previous cell
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else None

if df is not None and not df.empty:
    # Example: Find a numeric column, pick the first one that is detected as numeric
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_cols:
        # Try to convert columns to numeric
        inferred_numeric = []
        for col in df.columns:
            try:
                pd.to_numeric(df[col])
                inferred_numeric.append(col)
            except Exception:
                pass
        numeric_cols = inferred_numeric
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")
        # Make sure the col is float
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize (z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a non-numeric (presumably categorical) column
        non_numeric_cols = [c for c in df.columns if c not in numeric_cols]
        group_field = non_numeric_cols[0] if non_numeric_cols else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found to group by.")
    else:
        print("No numeric fields detected in dataframe.")
else:
    print("No data loaded or empty DataFrame.")

## 5. Visualization
Visualize the data distribution of the selected numeric field, either with a histogram or box plot. Update the field name according to your actual field name above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_cols:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by group_field if present
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to discover the structure of a Croissant-based schema, list available record sets and fields using their `@id`, and load, filter, and visualize data from the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset. This workflow can be adapted to other Croissant-based datasets for reproducible, schema-driven analyses.